# Rung 4 — possession-outcome step heads (Colab training notebook)

CLAUDE.md architecture ladder, rung 4. Scope per `NEXT_SESSION.md` item 4:
learned possession-outcome heads (NOT the DeepSets lineup encoder — a prior
experiment found lineup-archetype mix carried negligible signal). This
trains five heads sharing one MLP trunk: **outcome** (6-way), **zone**
(3-way, shot-attempt rows only), **make** (binary, shot-attempt rows only),
**rebound** (binary, missed-shot rows only), **duration** (regression,
seconds). The *shooter* head is out of scope here (needs the lineup
encoder).

**Self-contained**: every class/constant used below is defined inline in
this notebook (duplicated from, and must be kept in sync with,
`nba/features/possession_step_features.py` and
`nba/models/rung4_stepheads.py` in the main repo) so this notebook has no
dependency on cloning the private repo into Colab — only the exported
parquet file needs to be uploaded.

**Leakage safety** (do not change without re-reading
`nba/features/possession_step_features.py`'s module docstring): every
feature column is either (a) a strictly-prior as-of team rating
(`off_*_prior`/`def_*_prior`, computed from games *before* this one) or
(b) intra-game state known *before* this possession's own outcome
(`period`, `clock_start`, `score_diff`). Every label column
(`outcome`, `shot_zone`, `made_shot`, `oreb`, `duration_s`) is this
possession's own realized result — never used as a feature. The
train/holdout split below is chronological (by `game_date`), never random.

**Run order**: 1) run `nba/models/colab/export_training_data.py` locally
to produce `possession_steps.parquet`, 2) upload it to Drive or directly
to the Colab runtime, 3) set `PARQUET_PATH` below, 4) Runtime > Run all.
See `nba/models/colab/README.md` for the full walkthrough and how to pull
results back into the local registry.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
from torch import nn

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
# --- Config (edit before running) ---------------------------------------
SEED = 0
PARQUET_PATH = "possession_steps.parquet"  # uploaded export; see README
HIDDEN = 64
LR = 1e-3
N_EPOCHS = 20
BATCH_SIZE = 4096
# Frozen final holdout: the LAST full season in the export is never used
# for tuning (CLAUDE.md "frozen final holdout season"). Everything else is
# the tunable walk-forward set. Set explicitly once the real export's
# season range is known; left as None means "infer: the max season".
HOLDOUT_SEASON = None
OUT_DIR = Path("rung4_stepheads_artifacts")
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
# --- Constants (keep in sync with nba/features/possession_step_features.py) ---
OUTCOME_CLASSES = ["FGM2", "FGM3", "FGA_miss", "TOV", "FT_trip", "other"]
ZONE_CLASSES = ["rim", "mid", "above3"]
SHOT_ATTEMPT_OUTCOMES = ["FGM2", "FGM3", "FGA_miss"]
MADE_OUTCOMES = ["FGM2", "FGM3"]
POSSESSION_STEP_FEATURE_COLUMNS = [
    "period",
    "clock_start",
    "score_diff",
    "off_is_home",
    "off_off_rtg_prior",
    "off_def_rtg_prior",
    "off_pace_prior",
    "def_off_rtg_prior",
    "def_def_rtg_prior",
    "league_avg_ppp_asof",
]
DURATION_LOSS_WEIGHT = 0.001

In [ ]:
# --- Model (keep in sync with nba/models/rung4_stepheads.py) ------------
class StepHeadsNet(nn.Module):
    def __init__(self, n_features: int, hidden: int = 64) -> None:
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.outcome_head = nn.Linear(hidden, len(OUTCOME_CLASSES))
        self.zone_head = nn.Linear(hidden, len(ZONE_CLASSES))
        self.make_head = nn.Linear(hidden, 1)
        self.rebound_head = nn.Linear(hidden, 1)
        self.duration_head = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        h = self.trunk(x)
        return {
            "outcome_logits": self.outcome_head(h),
            "zone_logits": self.zone_head(h),
            "make_logit": self.make_head(h).squeeze(-1),
            "rebound_logit": self.rebound_head(h).squeeze(-1),
            "duration": self.duration_head(h).squeeze(-1),
        }


def seed_everything(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def feature_matrix(df: pl.DataFrame, mean=None, std=None):
    raw = df.select(POSSESSION_STEP_FEATURE_COLUMNS).to_numpy().astype(np.float64)
    mean = raw.mean(axis=0) if mean is None else mean
    std = raw.std(axis=0) if std is None else std
    std = np.where(std < 1e-8, 1.0, std)
    return (raw - mean) / std, mean, std


def frame_to_tensors(df: pl.DataFrame, mean=None, std=None):
    x, mean, std = feature_matrix(df, mean, std)
    outcome_to_idx = {c: i for i, c in enumerate(OUTCOME_CLASSES)}
    zone_to_idx = {c: i for i, c in enumerate(ZONE_CLASSES)}
    outcomes = df.get_column("outcome").to_list()
    outcome_idx = torch.tensor([outcome_to_idx[o] for o in outcomes], dtype=torch.long)
    is_shot = torch.tensor([o in SHOT_ATTEMPT_OUTCOMES for o in outcomes], dtype=torch.bool)
    is_miss = torch.tensor([o == "FGA_miss" for o in outcomes], dtype=torch.bool)
    zones = df.get_column("shot_zone").to_list()
    zone_idx = torch.tensor([zone_to_idx.get(z, 0) for z in zones], dtype=torch.long)
    made = torch.tensor(df.get_column("made_shot").to_numpy().astype(np.float32))
    oreb = torch.tensor(df.get_column("oreb").to_numpy().astype(np.float32))
    duration = torch.tensor(df.get_column("duration_s").to_numpy().astype(np.float32))
    tensors = {
        "x": torch.tensor(x, dtype=torch.float32),
        "outcome_idx": outcome_idx,
        "is_shot": is_shot,
        "zone_idx": zone_idx,
        "made": made,
        "is_miss": is_miss,
        "oreb": oreb,
        "duration": duration,
    }
    return tensors, mean, std


def compute_losses(net: StepHeadsNet, batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    out = net(batch["x"])
    zero = torch.tensor(0.0, device=batch["x"].device)
    outcome_loss = nn.functional.cross_entropy(out["outcome_logits"], batch["outcome_idx"])
    is_shot = batch["is_shot"]
    if is_shot.any():
        zone_loss = nn.functional.cross_entropy(
            out["zone_logits"][is_shot], batch["zone_idx"][is_shot]
        )
        make_loss = nn.functional.binary_cross_entropy_with_logits(
            out["make_logit"][is_shot], batch["made"][is_shot]
        )
    else:
        zone_loss, make_loss = zero, zero
    is_miss = batch["is_miss"]
    if is_miss.any():
        rebound_loss = nn.functional.binary_cross_entropy_with_logits(
            out["rebound_logit"][is_miss], batch["oreb"][is_miss]
        )
    else:
        rebound_loss = zero
    duration_loss = nn.functional.mse_loss(out["duration"], batch["duration"])
    total = (
        outcome_loss + zone_loss + make_loss + rebound_loss + DURATION_LOSS_WEIGHT * duration_loss
    )
    return {
        "outcome": outcome_loss,
        "zone": zone_loss,
        "make": make_loss,
        "rebound": rebound_loss,
        "duration": duration_loss,
        "total": total,
    }

In [ ]:
# --- Load the exported parquet + chronological (never random) split -----
df = pl.read_parquet(PARQUET_PATH)
print("rows:", df.height, "| date range:", df["game_date"].min(), "to", df["game_date"].max())

holdout_season = HOLDOUT_SEASON if HOLDOUT_SEASON is not None else df["season"].max()
tunable = df.filter(pl.col("season") < holdout_season)
holdout = df.filter(pl.col("season") == holdout_season)
print(
    f"tunable: {tunable.height} rows | "
    f"frozen holdout (season {holdout_season}): {holdout.height} rows"
)

# Walk-forward within the tunable set: last 15% of dates (by game_date) is
# the within-tunable validation window used for early stopping / logging
# only -- never the frozen holdout above, and never used to pick the final
# reported holdout metric.
dates = tunable.select("game_date").unique().sort("game_date").to_series()
cutoff = dates[int(len(dates) * 0.85)]
train_df = tunable.filter(pl.col("game_date") < cutoff)
val_df = tunable.filter(pl.col("game_date") >= cutoff)
print(f"train: {train_df.height} rows (< {cutoff}) | val: {val_df.height} rows (>= {cutoff})")

In [ ]:
# --- Train ----------------------------------------------------------------
seed_everything(SEED)
train_batch, feat_mean, feat_std = frame_to_tensors(train_df)
val_batch, _, _ = frame_to_tensors(val_df, feat_mean, feat_std)
train_batch = {k: v.to(DEVICE) for k, v in train_batch.items()}
val_batch = {k: v.to(DEVICE) for k, v in val_batch.items()}

net = StepHeadsNet(n_features=len(POSSESSION_STEP_FEATURE_COLUMNS), hidden=HIDDEN).to(DEVICE)
optimizer = torch.optim.Adam(net.parameters(), lr=LR)

n = train_batch["x"].shape[0]
history = []
start = time.monotonic()
for epoch in range(N_EPOCHS):
    net.train()
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for i in range(0, n, BATCH_SIZE):
        idx = perm[i : i + BATCH_SIZE]
        mini_batch = {k: v[idx] for k, v in train_batch.items()}
        optimizer.zero_grad()
        losses = compute_losses(net, mini_batch)
        losses["total"].backward()
        optimizer.step()
        epoch_loss += float(losses["total"].detach()) * idx.shape[0]
    epoch_loss /= n

    net.eval()
    with torch.no_grad():
        val_losses = compute_losses(net, val_batch)
    history.append(
        {"epoch": epoch, "train_loss": epoch_loss, "val_loss": float(val_losses["total"])}
    )
    print(f"epoch {epoch:3d} | train {epoch_loss:.4f} | val {float(val_losses['total']):.4f}")

print(f"training took {time.monotonic() - start:.1f}s")

In [ ]:
# --- Score the frozen holdout (reported, never used to pick hyperparams) ---
holdout_batch, _, _ = (
    frame_to_tensors(holdout, feat_mean, feat_std) if holdout.height else (None, None, None)
)
net.eval()
holdout_metrics = {}
if holdout_batch is not None and holdout.height:
    holdout_batch = {k: v.to(DEVICE) for k, v in holdout_batch.items()}
    with torch.no_grad():
        hl = compute_losses(net, holdout_batch)
    holdout_metrics = {k: float(v.detach()) for k, v in hl.items()}
    print("frozen holdout losses:", holdout_metrics)
else:
    print("no holdout rows (e.g. single-season export) -- holdout_metrics left empty")

In [ ]:
# --- Save artifacts (weights, config, metrics) ----------------------------
weights_path = OUT_DIR / "rung4_stepheads_weights.pt"
torch.save(net.state_dict(), weights_path)

config = {
    "model_name": "rung4_stepheads",
    "rung": 4,
    "seed": SEED,
    "hidden": HIDDEN,
    "lr": LR,
    "n_epochs": N_EPOCHS,
    "batch_size": BATCH_SIZE,
    "feature_columns": POSSESSION_STEP_FEATURE_COLUMNS,
    "feature_mean": feat_mean.tolist(),
    "feature_std": feat_std.tolist(),
    "outcome_classes": OUTCOME_CLASSES,
    "zone_classes": ZONE_CLASSES,
    "holdout_season": int(holdout_season) if holdout_season is not None else None,
    "n_train_rows": train_df.height,
    "n_val_rows": val_df.height,
    "n_holdout_rows": holdout.height,
}
metrics = {
    "history": history,
    "final_train_loss": history[-1]["train_loss"] if history else None,
    "final_val_loss": history[-1]["val_loss"] if history else None,
    "holdout": holdout_metrics,
}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

size_mb = weights_path.stat().st_size / 1e6
print(f"weights: {weights_path} ({size_mb:.2f} MB -- must stay under ~100MB per CLAUDE.md)")
print(f"config: {OUT_DIR / 'config.json'}")
print(f"metrics: {OUT_DIR / 'metrics.json'}")
print(
    "\nDownload the whole OUT_DIR, then see nba/models/colab/README.md"
    " 'Pulling results back into the local registry'."
)